In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

dim = 10

perf_dir = "/data1/home/jw1017/AS_BBO/AS_BBOB_SOO/data"
performance_path = Path(perf_dir) / f'performance_records_{dim}.csv'
iid_path = Path(perf_dir) / f'iid_perform_{dim}.csv'

if not performance_path.exists() or not iid_path.exists():
    raise FileNotFoundError(
        f'Missing CSVs under {Path(perf_dir).resolve()}. Run in terminal:\n'
        f'  python extract_bbob_iid_performance.py --outdir {perf_dir} --no-allow-missing\n'
        f'Expected: {performance_path} and {iid_path}'
    )

performance_df = pd.read_csv(performance_path)
iid_perform_df = pd.read_csv(iid_path)

print('Loaded:')
print('  performance_df:', performance_df.shape)
print('  iid_perform_df:', iid_perform_df.shape)
print('Algorithms:', sorted(performance_df['algo'].unique().tolist()))

Loaded:
  performance_df: (5760, 6)
  iid_perform_df: (480, 15)
Algorithms: ['BrentSTEPqi_Posik', 'BrentSTEPrr_Posik', 'CMA-CSA_Atamna', 'HCMA_loshchilov_noiseless', 'HMLSL_pal_noiseless', 'IPOP400D_auger_noiseless', 'MCS_huyer_noiseless', 'MLSL_pal_noiseless', 'OQNLP_pal_noiseless', 'SMAC-BBOB_hutter_noiseless', 'fmincon_pal_noiseless', 'fminunc_pal_noiseless']


In [2]:
display(performance_df.head())
display(iid_perform_df.head())

counts = (
    performance_df.groupby(['algo', 'Dim'])['funcId'].nunique()
    .unstack(fill_value=0)
    .sort_index()
)
display(counts)

succ_rate = performance_df.groupby('algo')['success'].mean().sort_values(ascending=False)
display(succ_rate)

,algo,funcId,Dim,iid,evals,success
0,BrentSTEPqi_Posik,1,2,1,11.0,True
1,BrentSTEPqi_Posik,1,2,2,12.0,True
2,BrentSTEPqi_Posik,1,2,3,11.0,True
3,BrentSTEPqi_Posik,1,2,4,9.0,True
4,BrentSTEPqi_Posik,1,2,5,10.0,True


,funcId,Dim,iid,BrentSTEPqi_Posik,BrentSTEPrr_Posik,CMA-CSA_Atamna,HCMA_loshchilov_noiseless,HMLSL_pal_noiseless,IPOP400D_auger_noiseless,MCS_huyer_noiseless,MLSL_pal_noiseless,OQNLP_pal_noiseless,SMAC-BBOB_hutter_noiseless,fmincon_pal_noiseless,fminunc_pal_noiseless
0,1,2,1,2.2,2.2,17.0,1.2,1.4,9.8,1.4,1.4,2.8,2.0,1.4,1.4
1,1,2,2,2.4,2.4,21.8,1.2,2.6,14.0,1.4,2.6,4.0,3.6,2.6,1.4
2,1,2,3,2.2,2.2,16.8,1.2,3.2,18.2,4.4,3.2,4.4,2.0,3.2,1.4
3,1,2,4,1.8,1.8,11.4,1.2,1.4,9.6,4.2,1.4,3.0,2.8,1.4,1.4
4,1,2,5,2.0,2.0,15.0,1.2,2.6,17.6,4.2,2.6,4.2,4.0,2.6,1.4


Dim,2,3,5,10
algo,,,,
BrentSTEPqi_Posik,24,24,24,24
BrentSTEPrr_Posik,24,24,24,24
CMA-CSA_Atamna,24,24,24,24
HCMA_loshchilov_noiseless,24,24,24,24
HMLSL_pal_noiseless,24,24,24,24
IPOP400D_auger_noiseless,24,24,24,24
MCS_huyer_noiseless,24,24,24,24
MLSL_pal_noiseless,24,24,24,24
OQNLP_pal_noiseless,24,24,24,24


algo
HCMA_loshchilov_noiseless     0.981250
CMA-CSA_Atamna                0.875000
HMLSL_pal_noiseless           0.831250
fmincon_pal_noiseless         0.668750
MLSL_pal_noiseless            0.664583
OQNLP_pal_noiseless           0.660417
fminunc_pal_noiseless         0.643750
MCS_huyer_noiseless           0.631250
IPOP400D_auger_noiseless      0.554167
BrentSTEPqi_Posik             0.356250
BrentSTEPrr_Posik             0.354167
SMAC-BBOB_hutter_noiseless    0.141667
Name: success, dtype: float64

In [3]:
# Recompute iid_perform (matches the old notebook logic) from performance_df
unique_instances = performance_df.drop_duplicates(subset=['algo', 'funcId', 'Dim', 'iid'])
group_success = (
    unique_instances.groupby(['algo', 'funcId', 'Dim'])['success']
    .sum()
    .reset_index()
    .rename(columns={'success': 'group_success_sum'})
)

merged_df = unique_instances.merge(group_success, on=['algo', 'funcId', 'Dim'], how='left')
merged_df['avg_evals'] = np.where(
    merged_df['group_success_sum'] > 0,
    merged_df['evals'] / merged_df['group_success_sum'],
    np.nan,
)

result_df = merged_df[['algo', 'funcId', 'Dim', 'iid', 'avg_evals']]
fill_value = float(np.nanmax(result_df['avg_evals'].to_numpy()))
result_df = result_df.fillna(fill_value)

iid_perform_recomputed = (
    result_df.pivot(index=['funcId', 'Dim', 'iid'], columns='algo', values='avg_evals')
    .reset_index()
)

display(iid_perform_recomputed.head())

# Optional: quick equality check (ignoring column order)
common_cols = ['funcId', 'Dim', 'iid'] + sorted(set(iid_perform_df.columns) & set(iid_perform_recomputed.columns) - {'funcId','Dim','iid'})
a = iid_perform_df[common_cols].sort_values(['funcId','Dim','iid']).reset_index(drop=True)
b = iid_perform_recomputed[common_cols].sort_values(['funcId','Dim','iid']).reset_index(drop=True)
print('Max abs diff (common cols):', np.nanmax(np.abs(a.drop(columns=['funcId','Dim','iid']).to_numpy() - b.drop(columns=['funcId','Dim','iid']).to_numpy())))

algo,funcId,Dim,iid,BrentSTEPqi_Posik,BrentSTEPrr_Posik,CMA-CSA_Atamna,HCMA_loshchilov_noiseless,HMLSL_pal_noiseless,IPOP400D_auger_noiseless,MCS_huyer_noiseless,MLSL_pal_noiseless,OQNLP_pal_noiseless,SMAC-BBOB_hutter_noiseless,fmincon_pal_noiseless,fminunc_pal_noiseless
0,1,2,1,2.2,2.2,17.0,1.2,1.4,9.8,1.4,1.4,2.8,2.0,1.4,1.4
1,1,2,2,2.4,2.4,21.8,1.2,2.6,14.0,1.4,2.6,4.0,3.6,2.6,1.4
2,1,2,3,2.2,2.2,16.8,1.2,3.2,18.2,4.4,3.2,4.4,2.0,3.2,1.4
3,1,2,4,1.8,1.8,11.4,1.2,1.4,9.6,4.2,1.4,3.0,2.8,1.4,1.4
4,1,2,5,2.0,2.0,15.0,1.2,2.6,17.6,4.2,2.6,4.2,4.0,2.6,1.4


Max abs diff (common cols): 2.9103830456733704e-11


In [ ]:
# Compute ERT + relERT tables from performance_df
unique_instances = performance_df.drop_duplicates(subset=['algo', 'funcId', 'Dim', 'iid']).copy()
grouped = unique_instances.groupby(['algo', 'funcId', 'Dim'], as_index=False).agg(
    sum_evals=('evals', 'sum'),
    n_success=('success', 'sum'),
)
grouped['ERT'] = np.where(grouped['n_success'] > 0, grouped['sum_evals'] / grouped['n_success'], np.nan)

df = grouped.rename(columns={'algo': 'Algo'})
df['Problem'] = df['funcId'].apply(lambda i: f'f{int(i)}')
df = df[['Algo', 'Problem', 'Dim', 'ERT']].copy()
df.replace([np.inf, -np.inf], np.nan, inplace=True)

df['relERT'] = df['ERT'] / df.groupby(['Problem', 'Dim'])['ERT'].transform('min')
df = df.fillna(36690.3)

order = [f'f{i}' for i in range(1, 25)]
df_wide = df.pivot(index=['Problem', 'Dim'], columns='Algo', values='relERT').loc[order]
display(df_wide.head())

df_wide.to_csv(f'relert_bbob_{dim}.csv')
print(f'Wrote original_{dim}.csv')

Algo         BrentSTEPqi_Posik  BrentSTEPrr_Posik  CMA-CSA_Atamna  \
Problem Dim                                                         
f1      2             1.766667           1.766667       13.666667   
        3             2.100000           2.100000       20.075000   
        5             2.350000           2.350000       21.116667   
        10            2.428571           2.428571       26.696429   
f2      2             1.000000           1.252427       17.213592   

Algo         HCMA_loshchilov_noiseless  HMLSL_pal_noiseless  \
Problem Dim                                                   
f1      2                      1.00000             1.866667   
        3                      1.00000             1.925000   
        5                      1.00000             2.183333   
        10                     1.00000             2.205357   
f2      2                      9.23301             1.951456   

Algo         IPOP400D_auger_noiseless  MCS_huyer_noiseless  \
Problem Dim                                                  
f1      2                   11.533333             2.600000   
        3                   16.525000             2.375000   
        5                   20.716667             2.633333   
        10                  22.589286             2.232143   
f2      2                   22.310680             3.155340   

Algo         MLSL_pal_noiseless  OQNLP_pal_noiseless  \
Problem Dim                                            
f1      2              1.866667             3.066667   
        3              1.925000             2.875000   
        5              2.183333             2.516667   
        10             2.205357             1.973214   
f2      2              1.951456             2.475728   

Algo         SMAC-BBOB_hutter_noiseless  fmincon_pal_noiseless  \
Problem Dim                                                      
f1      2                      2.400000               1.866667   
        3                      3.450000               1.925000   
        5                      3.300000               2.183333   
        10                     6.544643               2.205357   
f2      2                  36690.300000               1.951456   

Algo         fminunc_pal_noiseless  
Problem Dim                         
f1      2                 1.166667  
        3                 1.125000  
        5                 1.083333  
        10                1.026786  
f2      2                 4.475728

Wrote original_10.csv


In [ ]:
df_wide

Algo         BrentSTEPqi_Posik  BrentSTEPrr_Posik  CMA-CSA_Atamna  \
Problem Dim                                                         
f1      2             1.766667           1.766667       13.666667   
        3             2.100000           2.100000       20.075000   
        5             2.350000           2.350000       21.116667   
        10            2.428571           2.428571       26.696429   
f2      2             1.000000           1.252427       17.213592   
...                        ...                ...             ...   
f23     10        36690.300000       36690.300000       82.495922   
f24     2            12.158346       36690.300000    36690.300000   
        3         36690.300000       36690.300000    36690.300000   
        5         36690.300000       36690.300000    36690.300000   
        10        36690.300000       36690.300000    36690.300000   

Algo         HCMA_loshchilov_noiseless  HMLSL_pal_noiseless  \
Problem Dim                                                   
f1      2                     1.000000             1.866667   
        3                     1.000000             1.925000   
        5                     1.000000             2.183333   
        10                    1.000000             2.205357   
f2      2                     9.233010             1.951456   
...                                ...                  ...   
f23     10                    1.000000         36690.300000   
f24     2                   116.126504            24.819256   
        3                   425.928214             6.024221   
        5                     1.000000         36690.300000   
        10                    1.000000         36690.300000   

Algo         IPOP400D_auger_noiseless  MCS_huyer_noiseless  \
Problem Dim                                                  
f1      2                   11.533333             2.600000   
        3                   16.525000             2.375000   
        5                   20.716667             2.633333   
        10                  22.589286             2.232143   
f2      2                   22.310680             3.155340   
...                               ...                  ...   
f23     10               36690.300000         36690.300000   
f24     2                36690.300000             8.059043   
        3                36690.300000         36690.300000   
        5                36690.300000         36690.300000   
        10               36690.300000         36690.300000   

Algo         MLSL_pal_noiseless  OQNLP_pal_noiseless  \
Problem Dim                                            
f1      2              1.866667             3.066667   
        3              1.925000             2.875000   
        5              2.183333             2.516667   
        10             2.205357             1.973214   
f2      2              1.951456             2.475728   
...                         ...                  ...   
f23     10         36690.300000         36690.300000   
f24     2              7.511172             1.000000   
        3          36690.300000             1.000000   
        5          36690.300000         36690.300000   
        10         36690.300000         36690.300000   

Algo         SMAC-BBOB_hutter_noiseless  fmincon_pal_noiseless  \
Problem Dim                                                      
f1      2                      2.400000               1.866667   
        3                      3.450000               1.925000   
        5                      3.300000               2.183333   
        10                     6.544643               2.205357   
f2      2                  36690.300000               1.951456   
...                                 ...                    ...   
f23     10                 36690.300000           36690.300000   
f24     2                  36690.300000              17.493551   
        3                  36690.300000           36690.300000   
        5           

In [10]:
# Check the SBS mean, median, p90
sbs = int(np.argmin(np.mean(df_wide.to_numpy(), axis=0)))
sbs

3

In [19]:
# mean median p90
print(df_wide.to_numpy()[:, sbs].mean(), np.median(df_wide.to_numpy()[:, sbs]), np.percentile(df_wide.to_numpy()[:, sbs], 90))

86.37072271151112 3.1795592040268668 9.883505885026434


In [30]:
np.mean(df_wide.to_numpy(), axis=0)

array([16945.55603565, 17723.1826612 ,  3496.17446976,    86.37072271,
        3075.70143027, 11094.47561142, 11189.08195418, 10341.38008837,
        9180.46749605, 29811.63340732,  9580.2359004 , 10740.39988544])

In [31]:
df_deepela = pd.read_csv("../../data/relert_bbob_10.csv")
sbs = int(np.argmin(np.mean(df_deepela.iloc[:, 2:].to_numpy(), axis=0)))
print(df_deepela.to_numpy()[:, sbs+2].mean(), np.median(df_deepela.to_numpy()[:, sbs+2]), np.percentile(df_deepela.to_numpy()[:, sbs+2], 90))
print(np.mean(df_deepela.iloc[:, 2:].to_numpy(), axis=0))

30.37341725601549 3.4384216532384433 12.476501395664348
[16873.29991715 17644.44007437  3466.01218941    30.37341726
  3064.26006516 11091.93844731 11151.01633978 10328.80683446
  9177.88443745 29811.44691309  9567.36552889 10718.84408891]


In [ ]:
# Check whether original_{dim}.csv with /data1/home/jw1017/AS_BBO/AS_BBOB_SOO/data/relert_bbob_{dim}.csv are identical
relert_df = pd.read_csv(Path(perf_dir) / f'relert_bbob_{dim}.csv', index_col=['Problem', 'Dim'])
relert_aligned = relert_df.rename(columns={"M_LSL_pal_noiseless": "MLSL_pal_noiseless"})
relert_aligned = relert_aligned[df_wide.columns]

same_index = relert_aligned.index.equals(df_wide.index)
same_shape = relert_aligned.shape == df_wide.shape
max_diff = np.nanmax(np.abs(relert_aligned.to_numpy() - df_wide.to_numpy()))

print(f"Same index: {same_index}")
print(f"Same shape: {same_shape}")
print(f"Max abs diff: {max_diff}")
print("Identical (within floating tolerance):", same_index and same_shape and np.isclose(max_diff, 0.0))

# Identify rows and columns with differences
mask_np = ~np.isclose(relert_aligned.to_numpy(), df_wide.to_numpy(), equal_nan=True)
diff_mask = pd.DataFrame(mask_np, index=relert_aligned.index, columns=relert_aligned.columns)
rows_with_diff = diff_mask.any(axis=1)

print("Rows with differences:")
for idx in relert_aligned.index[rows_with_diff]:
    cols = diff_mask.loc[idx]
    differing_cols = cols[cols].index.tolist()
    print(f"{idx} differs in columns: {differing_cols}")

display((relert_aligned - df_wide).where(diff_mask).loc[rows_with_diff])

Same index: True
Same shape: True
Max abs diff: 2766.239754098361
Identical (within floating tolerance): False
Rows with differences:


AttributeError: 'numpy.ndarray' object has no attribute 'loc'